# Tarea 6. Dataset limpio documentado
## Análisis histórico del Tour de Francia

**Asignatura:** Programación avanzada para ciencia de datos
**Unidad:** 2. Manejo preliminar de datos
**Fuente:** LeTourDataSet — Thomas Camminady
**Repositorio:** https://github.com/thomascamminady/LeTourDataSet

## Preparación de los datos
## Descarga de los cuatro CSV
Solo si faltan archivos locales

In [1]:
# Importamos Path para trabajar con carpetas y archivos.
from pathlib import Path

# Importamos urlopen para descargar los archivos originales.
from urllib.request import urlopen

# Definimos la carpeta donde guardaremos los datos de la Tarea 6.
carpeta_datos = Path("datos_tarea6")

# Creamos la carpeta si todavía no existe.
carpeta_datos.mkdir(parents=True, exist_ok=True)

# Dirección base del repositorio de GitHub.
url_base = (
    "https://raw.githubusercontent.com/"
    "thomascamminady/LeTourDataSet/master/data/"
)

# Definimos los cuatro archivos originales y sus rutas.
archivos = {
    "Ciclistas masculinos": "men/TDF_Riders_History.csv",
    "Etapas masculinas": "men/TDF_Stages_History.csv",
    "Ciclistas femeninos": "women/TDFF_Riders_History.csv",
    "Etapas femeninas": "women/TDFF_Stages_History.csv"
}

# Recorremos los cuatro archivos definidos anteriormente.
for nombre, ruta in archivos.items():

    # Construimos la dirección completa del archivo en GitHub.
    url = url_base + ruta

    # Obtenemos únicamente el nombre del archivo CSV.
    nombre_archivo = Path(ruta).name

    # Definimos dónde se guardará el archivo localmente.
    destino = carpeta_datos / nombre_archivo

    # Descargamos el archivo solamente si todavía no existe.
    if not destino.is_file():

        # Obtenemos los bytes originales desde GitHub.
        with urlopen(url, timeout=30) as respuesta:
            contenido = respuesta.read()

        # Guardamos el CSV original sin modificarlo.
        destino.write_bytes(contenido)

        print(f"Descargado: {nombre_archivo}")

    else:
        # Si ya existe, evitamos descargarlo nuevamente.
        print(f"Ya existe: {nombre_archivo}")

    # Comprobamos que el archivo existe y no está vacío.
    assert destino.is_file()
    assert destino.stat().st_size > 0

    # Informamos que se guardó correctamente.
    print(f"Guardado: {destino} ({destino.stat().st_size:,} bytes)")

print("\nLos cuatro archivos originales se guardaron correctamente.")

Ya existe: TDF_Riders_History.csv
Guardado: datos_tarea6\TDF_Riders_History.csv (1,012,996 bytes)
Ya existe: TDF_Stages_History.csv
Guardado: datos_tarea6\TDF_Stages_History.csv (230,538 bytes)
Ya existe: TDFF_Riders_History.csv
Guardado: datos_tarea6\TDFF_Riders_History.csv (59,315 bytes)
Ya existe: TDFF_Stages_History.csv
Guardado: datos_tarea6\TDFF_Stages_History.csv (6,003 bytes)

Los cuatro archivos originales se guardaron correctamente.


## Diagnóstico inicial
Lee los cuatro CSV locales y crea datasets

In [2]:

# Importamos pandas para leer y analizar los archivos CSV.
import pandas as pd

# Leemos cada CSV y lo guardamos en un diccionario de DataFrames.

# Leemos los archivos CSV desde la carpeta local.
# Path(ruta).name obtiene solo el nombre del archivo,
# eliminando las carpetas "men/" y "women/" de la ruta.
datasets = {
    nombre: pd.read_csv(carpeta_datos / Path(ruta).name)
    for nombre, ruta in archivos.items()
}

# Elaboramos el diagnóstico inicial de los datasets.
# Construimos una tabla de diagnóstico sin modificar los datos.
diagnostico = pd.DataFrame([
    {
        "Dataset": nombre,
        "Filas": len(df),
        "Columnas": df.shape[1],
        "Nulos totales": int(df.isna().sum().sum()),
        "Filas con nulos": int(df.isna().any(axis=1).sum()),
        "Duplicados exactos": int(df.duplicated().sum())
    }
    for nombre, df in datasets.items()
])

# Mostramos los resultados.
display(diagnostico)

,Dataset,Filas,Columnas,Nulos totales,Filas con nulos,Duplicados exactos
0,Ciclistas masculinos,10337,14,20310,10316,0
1,Etapas masculinas,2434,11,7856,2434,0
2,Ciclistas femeninos,581,14,1073,579,0
3,Etapas femeninas,42,10,0,0,0


## Verificar los CSV locales
Esta celda verifica que los archivos existan, no estén vacíos, tengan las dimensiones esperadas y puedan leerse sin diferencias respecto a los DataFrames del diagnóstico inicial.

In [3]:
# Creamos un diccionario para registrar las verificaciones.
verificaciones = {}

# Definimos las dimensiones esperadas de los CSV originales.
dimensiones_esperadas = {
    "Ciclistas masculinos": (10337, 14),
    "Etapas masculinas": (2434, 11),
    "Ciclistas femeninos": (581, 14),
    "Etapas femeninas": (42, 10)
}

# Recorremos los cuatro archivos definidos en la celda de descarga.
for nombre, ruta in archivos.items():

    # Construimos la ruta del archivo local.
    ruta_local = carpeta_datos / Path(ruta).name

    # Comprobamos que el archivo existe y no está vacío.
    assert ruta_local.is_file()
    assert ruta_local.stat().st_size > 0

    # Leemos nuevamente el CSV para verificar su contenido.
    df_verificacion = pd.read_csv(ruta_local)

    # Comprobamos que tiene las dimensiones esperadas.
    assert df_verificacion.shape == dimensiones_esperadas[nombre]

    # Comparamos los datos leídos con el diagnóstico inicial.
    pd.testing.assert_frame_equal(
        df_verificacion,
        datasets[nombre]
    )

    # Registramos el resultado de la verificación.
    verificaciones[nombre] = "CORRECTO"

    # Mostramos el resultado.
    print(
        f"{nombre}: {df_verificacion.shape[0]} filas, "
        f"{df_verificacion.shape[1]} columnas — CORRECTO"
    )

# Confirmamos que los cuatro archivos pasaron las pruebas.
print("\nLos cuatro CSV locales fueron verificados correctamente.")

Ciclistas masculinos: 10337 filas, 14 columnas — CORRECTO
Etapas masculinas: 2434 filas, 11 columnas — CORRECTO
Ciclistas femeninos: 581 filas, 14 columnas — CORRECTO
Etapas femeninas: 42 filas, 10 columnas — CORRECTO

Los cuatro CSV locales fueron verificados correctamente.


## Leer archivos locales
Dimensiones de ciclistas y etapas

In [4]:
# Recuperamos los dos DataFrames principales desde
# el diccionario que ya cargamos en el diagnóstico inicial.
ciclistas = datasets["Ciclistas masculinos"]
etapas = datasets["Etapas masculinas"]

# Comprobamos que conservan las dimensiones esperadas.
assert ciclistas.shape == (10337, 14)
assert etapas.shape == (2434, 11)

# Mostramos una confirmación de lectura.
print("Datos locales disponibles correctamente.")
print(f"Ciclistas masculinos: {ciclistas.shape}")
print(f"Etapas masculinas: {etapas.shape}")

Datos locales disponibles correctamente.
Ciclistas masculinos: (10337, 14)
Etapas masculinas: (2434, 11)


## Diagnóstico por columna
Analiza tipos, nulos y valores únicos

In [5]:
# Seleccionamos los dos datasets masculinos para analizarlos.
ciclistas = datasets["Ciclistas masculinos"]
etapas = datasets["Etapas masculinas"]

# Recorremos ambos DataFrames para diagnosticar sus columnas.
for nombre, df in [
    ("CICLISTAS MASCULINOS", ciclistas),
    ("ETAPAS MASCULINAS", etapas)
]:

    # Mostramos el nombre del dataset que estamos revisando.
    print(f"\n{'=' * 55}")
    print(nombre)
    print(f"{'=' * 55}")

    # Construimos una tabla con información de cada columna.
    diagnostico_columnas = pd.DataFrame({
        "Tipo de dato": df.dtypes.astype(str),
        "Valores nulos": df.isna().sum(),
        "Porcentaje nulos": (
            df.isna().mean() * 100
        ).round(2),
        "Valores únicos": df.nunique(dropna=True)
    })

    # Mostramos el diagnóstico completo.
    display(diagnostico_columnas)

    # Mostramos cinco registros para entender la estructura.
    print("\nPrimeros cinco registros:")
    display(df.head())


CICLISTAS MASCULINOS


,Tipo de dato,Valores nulos,Porcentaje nulos,Valores únicos
Rank,int64,0,0.00,174
Rider,str,0,0.00,3646
Rider No.,int64,0,0.00,262
Team,str,0,0.00,702
Times,str,0,0.00,9896
Gap,str,0,0.00,7642
B,str,10040,97.13,39
P,str,10270,99.35,6
Year,int64,0,0.00,113
Distance (km),int64,0,0.00,109



Primeros cinco registros:


,Rank,Rider,Rider No.,Team,Times,Gap,B,P,Year,Distance (km),Number of stages,ResultType,TotalSeconds,GapSeconds
0,1,MAURICE GARIN,1,TDF 1903 ***,94h 33' 14'',-,NaN,NaN,1903,2428,6,time,340394,0
1,2,LUCIEN POTHIER,37,TDF 1903 ***,97h 32' 35'',+ 02h 59' 21'',NaN,NaN,1903,2428,6,time,351155,10761
2,3,FERNAND AUGEREAU,39,TDF 1903 ***,99h 02' 38'',+ 04h 29' 24'',NaN,NaN,1903,2428,6,time,356558,16164
3,4,RODOLPHE MULLER,33,TDF 1903 ***,99h 12' 44'',+ 04h 39' 30'',NaN,NaN,1903,2428,6,time,357164,16770
4,5,JEAN-BAPTISTE FISCHER,12,TDF 1903 ***,99h 41' 58'',+ 05h 08' 44'',NaN,NaN,1903,2428,6,time,358918,18524



ETAPAS MASCULINAS


,Tipo de dato,Valores nulos,Porcentaje nulos,Valores únicos
Year,int64,0,0.00,113
TotalTDFDistance,int64,0,0.00,109
Stages,float64,0,0.00,80
Start,str,0,0.00,645
End,str,0,0.00,547
Winner of stage,str,34,1.40,1355
Yellow Jersey,str,243,9.98,285
Green jersey,str,1757,72.19,77
Polka-dot jersey,str,1760,72.31,141
White jersey,str,1783,73.25,100



Primeros cinco registros:


,Year,TotalTDFDistance,Stages,Start,End,Winner of stage,Yellow Jersey,Green jersey,Polka-dot jersey,White jersey,Leader
0,1903,2428,1.0,Paris,Lyon,Maurice Garin (Tdf 1903 ***),NaN,NaN,NaN,NaN,MAURICE GARIN
1,1903,2428,2.0,Lyon,Marseille,Hippolyte Aucouturier (Tdf 1903 ***),NaN,NaN,NaN,NaN,MAURICE GARIN
2,1903,2428,3.0,Marseille,Toulouse,Hippolyte Aucouturier (Tdf 1903 ***),NaN,NaN,NaN,NaN,MAURICE GARIN
3,1903,2428,4.0,Toulouse,Bordeaux,Charles Laeser (Tdf 1903 ***),NaN,NaN,NaN,NaN,MAURICE GARIN
4,1903,2428,5.0,Bordeaux,Nantes,Maurice Garin (Tdf 1903 ***),NaN,NaN,NaN,NaN,MAURICE GARIN


## Diagnóstico de normalización de campos de texto

Revisamos el archivo original antes de corregir las variantes de nombres de equipos.

In [6]:
# Seleccionamos las columnas que contienen datos de texto.
columnas_texto = ciclistas.select_dtypes(
    include=["object", "string", "str"]
).columns

# Guardaremos los resultados del diagnóstico en una lista.
resultados_texto = []

# Revisamos cada columna de texto por separado.
for columna in columnas_texto:

    # Excluimos los valores faltantes para evitar comparaciones incorrectas.
    serie = ciclistas[columna].dropna().astype("string")

    # Identificamos valores con espacios al principio o al final.
    espacios_externos = (serie != serie.str.strip()).sum()

    # Contamos los valores distintos antes de normalizar.
    unicos_originales = serie.nunique()

    # Simulamos una normalización básica sin modificar el dataset.
    serie_normalizada = serie.str.strip().str.casefold()

    # Contamos los valores distintos después de la normalización simulada.
    unicos_normalizados = serie_normalizada.nunique()

    # Guardamos los resultados de esta columna.
    resultados_texto.append({
        "Columna": columna,
        "Espacios_externos": int(espacios_externos),
        "Unicos_originales": int(unicos_originales),
        "Unicos_normalizados": int(unicos_normalizados)
    })

# Convertimos los resultados en un DataFrame.
diagnostico_texto = pd.DataFrame(resultados_texto)

# Mostramos el diagnóstico.
display(diagnostico_texto)

,Columna,Espacios_externos,Unicos_originales,Unicos_normalizados
0,Rider,0,3646,3646
1,Team,0,702,690
2,Times,0,9896,9896
3,Gap,0,7642,7642
4,B,0,39,39
5,P,0,6,6
6,ResultType,0,3,3


### Identificar los nombres de equipos afectados

In [7]:
# Seleccionamos los nombres de equipos, excluyendo valores faltantes.
equipos = ciclistas["Team"].dropna().astype("string")

# Obtenemos los nombres únicos originales.
equipos_unicos = pd.DataFrame({
    "Nombre_original": equipos.unique()
})

# Creamos una versión normalizada para comparar nombres.
# No modificamos la columna original del dataset.
equipos_unicos["Nombre_normalizado"] = (
    equipos_unicos["Nombre_original"]
    .str.strip()
    .str.casefold()
)

# Identificamos los nombres que comparten una misma
# representación normalizada.
variantes_equipos = equipos_unicos[
    equipos_unicos.duplicated(
        subset=["Nombre_normalizado"],
        keep=False
    )
].sort_values(
    ["Nombre_normalizado", "Nombre_original"]
)

# Mostramos cuántos grupos tienen variantes de escritura.
print(
    "Grupos de nombres con variantes:",
    variantes_equipos["Nombre_normalizado"].nunique()
)

# Mostramos los nombres originales para investigar
# si realmente corresponden al mismo equipo.
display(variantes_equipos)

Grupos de nombres con variantes: 12


,Nombre_original,Nombre_normalizado
68,BELGIQUE,belgique
104,Belgique,belgique
69,ESPAGNE,espagne
106,Espagne,espagne
510,FDJEUX.COM,fdjeux.com
498,FDJeux.com,fdjeux.com
425,FESTINA WATCHES,festina watches
446,FESTINA Watches,festina watches
65,FRANCE,france
103,France,france


## Análisis exploratorio de variables numéricas
Analizar los valores numéricos y posibles anomalías

In [8]:

# Analizamos las variables numéricas de los dos datasets.
for nombre, df in [
    ("CICLISTAS MASCULINOS", ciclistas),
    ("ETAPAS MASCULINAS", etapas)
]:

    # Identificamos las columnas con datos numéricos.
    columnas_numericas = df.select_dtypes(
        include="number"
    ).columns.tolist()

    # Mostramos qué columnas serán analizadas.
    print(f"\n{'=' * 55}")
    print(nombre)
    print(f"{'=' * 55}")
    print("Columnas numéricas:", columnas_numericas)

    # Calculamos estadísticas descriptivas:
    # cantidad, media, desviación, mínimo, cuartiles y máximo.
    estadisticas = df[columnas_numericas].describe().T

    # Mostramos los resultados redondeados.
    display(estadisticas.round(2))

    # Contamos los valores negativos en cada columna.
    # No los eliminamos: primero necesitamos interpretarlos.
    negativos = (df[columnas_numericas] < 0).sum()

    print("\nCantidad de valores negativos:")
    display(negativos.to_frame("Valores negativos"))



CICLISTAS MASCULINOS
Columnas numéricas: ['Rank', 'Rider No.', 'Year', 'Distance (km)', 'Number of stages', 'TotalSeconds', 'GapSeconds']


,count,mean,std,min,25%,50%,75%,max
Rank,10337.0,59.02,41.56,1.0,24.0,51.0,88.0,174.0
Rider No.,10337.0,90.95,61.02,1.0,37.0,85.0,136.0,321.0
Year,10337.0,1984.76,30.83,1903.0,1965.0,1991.0,2010.0,2026.0
Distance (km),10337.0,3918.00,570.56,2428.0,3483.0,3765.0,4231.0,5745.0
Number of stages,10337.0,22.12,2.92,6.0,21.0,21.0,24.0,32.0
TotalSeconds,10337.0,386355.59,136813.81,0.0,317171.0,342028.0,424142.0,1078413.0
GapSeconds,10337.0,9708.50,13310.98,0.0,4051.0,7677.0,11251.0,178645.0



Cantidad de valores negativos:


,Valores negativos
Rank,0
Rider No.,0
Year,0
Distance (km),0
Number of stages,0
TotalSeconds,0
GapSeconds,0



ETAPAS MASCULINAS
Columnas numéricas: ['Year', 'TotalTDFDistance', 'Stages']


,count,mean,std,min,25%,50%,75%,max
Year,2434.0,1970.37,33.18,1903.0,1947.0,1972.0,1998.0,2026.0
TotalTDFDistance,2434.0,4157.78,639.45,2428.0,3630.0,4098.0,4498.0,5745.0
Stages,2434.0,10.65,6.30,0.0,5.0,10.1,16.0,25.0



Cantidad de valores negativos:


,Valores negativos
Year,0
TotalTDFDistance,0
Stages,0


## Investigación de valores cero
Distinguir entre valores atípicos reales, valores válidos y posibles errores de calidad.

In [9]:

# Buscamos ciclistas cuyo tiempo total sea exactamente cero.
ciclistas_tiempo_cero = ciclistas[
    ciclistas["TotalSeconds"] == 0
]

# Buscamos registros de etapas cuyo número sea cero.
etapas_numero_cero = etapas[
    etapas["Stages"] == 0
]

# Mostramos cuántos registros cumplen cada condición.
print(
    "Ciclistas con TotalSeconds = 0:",
    len(ciclistas_tiempo_cero)
)

print(
    "Registros de etapas con Stages = 0:",
    len(etapas_numero_cero)
)

# Mostramos algunos registros para investigar su significado.
print("\nCiclistas con tiempo total igual a cero:")
display(ciclistas_tiempo_cero.head(10))

print("\nEtapas con valor cero:")
display(etapas_numero_cero.head(10))

# Identificamos los años en los que aparecen esos valores.
print("\nAños con TotalSeconds = 0:")
display(
    ciclistas_tiempo_cero["Year"]
    .value_counts()
    .sort_index()
)

print("\nAños con Stages = 0:")
display(
    etapas_numero_cero["Year"]
    .value_counts()
    .sort_index()
)


Ciclistas con TotalSeconds = 0: 271
Registros de etapas con Stages = 0: 41

Ciclistas con tiempo total igual a cero:


,Rank,Rider,Rider No.,Team,Times,Gap,B,P,Year,Distance (km),Number of stages,ResultType,TotalSeconds,GapSeconds
36,1,LOUIS TROUSSELIER,14,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
37,2,HIPPOLYTE AUCOUTURIER,13,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
38,3,JEAN-BAPTISTE DORTIGNACQ,31,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
39,4,EMILE GEORGET,50,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
40,5,LUCIEN MAZAN DIT PETIT-BRETON,77,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
41,6,AUGUSTINN RINGEVAL,22,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
42,7,PAUL CHAUVET,52,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
43,8,PHILIPPE PAUTRAT,7,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
44,9,JULIEN GABORY,21,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0
45,10,JULIEN MAITRON,54,TDF 1905 ***,00h 00' 00'',-,NaN,NaN,1905,2994,11,no-results,0,0



Etapas con valor cero:


,Year,TotalTDFDistance,Stages,Start,End,Winner of stage,Yellow Jersey,Green jersey,Polka-dot jersey,White jersey,Leader
1067,1967,4779,0.0,Angers,Angers,José-Maria Errandonea (Espana),José-Maria Errandonea,NaN,NaN,NaN,NaN
1092,1968,4492,0.0,Vittel,Vittel,Charly Grosskost (France B),Charly Grosskost,NaN,NaN,NaN,NaN
1118,1969,4117,0.0,Roubaix,Roubaix,Rudi Altig (Salvarani),Rudi Altig,NaN,NaN,NaN,NaN
1144,1970,4254,0.0,Limoges,Limoges,Eddy Merckx (Faema-Faemino),Eddy Merckx,NaN,NaN,NaN,NaN
1173,1971,3608,0.0,Mulhouse,Mulhouse,NaN,Julien Stevens,NaN,NaN,NaN,NaN
1198,1972,3846,0.0,Angers,Angers,Eddy Merckx (Molteni),Eddy Merckx,NaN,NaN,NaN,NaN
1223,1973,4090,0.0,Scheveningen,Scheveningen,Joop Zoetemelk (Gitane-Frigecreme),Joop Zoetemelk,NaN,NaN,NaN,NaN
1250,1974,4098,0.0,Brest,Brest,Eddy Merckx (Molteni),Eddy Merckx,NaN,NaN,NaN,NaN
1277,1975,4000,0.0,Charleroi,Charleroi,Francesco Moser (Filotex),Francesco Moser,NaN,NaN,NaN,NaN
1302,1976,4017,0.0,Merlin-Plage,Saint-Jean-de-Monts,Freddy Maertens (Flandria-Velda-Vleesbdrijf),Freddy Maertens,NaN,NaN,NaN,NaN



Años con TotalSeconds = 0:


Year
1905    24
1906    14
1907    33
1908    36
1909    54
1910    41
1911    28
1912    41
Name: count, dtype: int64


Años con Stages = 0:


Year
1967    1
1968    1
1969    1
1970    1
1971    1
1972    1
1973    1
1974    1
1975    1
1976    1
1977    1
1978    1
1979    1
1980    1
1981    1
1982    1
1983    1
1984    1
1985    1
1986    1
1987    1
1988    1
1989    1
1990    1
1991    1
1992    1
1993    1
1994    1
1995    1
1996    1
1997    1
1998    1
1999    1
2001    1
2002    1
2003    1
2004    1
2006    1
2007    1
2010    1
2012    1
Name: count, dtype: int64

## Validación de tiempos no disponibles

In [10]:
# Contamos los tipos de resultado existentes en el dataset.
print("Tipos de resultado:")
display(ciclistas["ResultType"].value_counts(dropna=False))

# Contamos cuántos registros tienen tiempo cero
# dentro de cada tipo de resultado.
print("\nTiempos iguales a cero por tipo de resultado:")

resumen_tiempos = (
    ciclistas
    .groupby("ResultType")
    .agg(
        Total_registros=("TotalSeconds", "size"),
        Tiempos_cero=("TotalSeconds", lambda x: (x == 0).sum()),
        Tiempos_positivos=("TotalSeconds", lambda x: (x > 0).sum())
    )
)

display(resumen_tiempos)

# Identificamos posibles excepciones:
# registros con tiempo cero que NO sean "no-results".
excepciones = ciclistas[
    (ciclistas["TotalSeconds"] == 0) &
    (ciclistas["ResultType"] != "no-results")
]

print("\nExcepciones encontradas:", len(excepciones))
display(excepciones.head(10))

Tipos de resultado:


ResultType
time          10066
points          197
no-results       74
Name: count, dtype: int64


Tiempos iguales a cero por tipo de resultado:


,Total_registros,Tiempos_cero,Tiempos_positivos
ResultType,,,
no-results,74,74,0
points,197,197,0
time,10066,0,10066



Excepciones encontradas: 197


,Rank,Rider,Rider No.,Team,Times,Gap,B,P,Year,Distance (km),Number of stages,ResultType,TotalSeconds,GapSeconds
74,1,LUCIEN MAZAN DIT PETIT-BRETON,12,TDF 1907 ***,47h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
75,2,GUSTAVE GARRIGOU,71,TDF 1907 ***,66h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
76,3,EMILE GEORGET,73,TDF 1907 ***,74h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
77,4,GEORGES PASSERIEU,70,TDF 1907 ***,85h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
78,5,FRANÇOIS BEAUGENDRE,87,TDF 1907 ***,123h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
79,6,EBERADO PAVESI,43,TDF 1907 ***,150h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
80,7,FRANÇOIS FABER,4,TDF 1907 ***,156h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
81,8,AUGUSTINN RINGEVAL,2,TDF 1907 ***,184h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
82,9,ALOIS CATTEAU,89,TDF 1907 ***,189h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0
83,10,FERDINAND PAYAN,18,TDF 1907 ***,227h 00' 00'',-,NaN,NaN,1907,4488,14,points,0,0


## Detección de valores atípicos mediante el método IQR

Se estudian las distancias del dataset original antes de decidir si es necesario modificarlas.

In [11]:
# Seleccionamos la columna de distancia del dataset original.
distancias = ciclistas["Distance (km)"].dropna()

# Calculamos el primer y tercer cuartil.
q1 = distancias.quantile(0.25)
q3 = distancias.quantile(0.75)

# Calculamos el rango intercuartílico.
iqr = q3 - q1

# Establecemos los límites para detectar valores atípicos.
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Identificamos las distancias que quedan fuera de los límites.
atipicos = distancias[
    (distancias < limite_inferior) |
    (distancias > limite_superior)
]

# Mostramos los resultados del análisis.
print(f"Primer cuartil (Q1): {q1:.2f} km")
print(f"Tercer cuartil (Q3): {q3:.2f} km")
print(f"Rango intercuartílico: {iqr:.2f} km")
print(f"Límite inferior: {limite_inferior:.2f} km")
print(f"Límite superior: {limite_superior:.2f} km")
print(f"Valores atípicos detectados: {len(atipicos)}")

Primer cuartil (Q1): 3483.00 km
Tercer cuartil (Q3): 4231.00 km
Rango intercuartílico: 748.00 km
Límite inferior: 2361.00 km
Límite superior: 5353.00 km
Valores atípicos detectados: 417


### Investigación de los valores atípicos por año

In [12]:
# Seleccionamos los registros completos cuyas distancias
# están fuera de los límites calculados mediante IQR.
registros_atipicos = ciclistas.loc[
    (ciclistas["Distance (km)"] < limite_inferior) |
    (ciclistas["Distance (km)"] > limite_superior)
]

# Agrupamos por año y distancia para identificar las
# ediciones históricas que presentan valores atípicos.
resumen_atipicos = (
    registros_atipicos
    .groupby(["Year", "Distance (km)"])
    .size()
    .reset_index(name="Cantidad_ciclistas")
    .sort_values("Year")
)

# Mostramos cuántas combinaciones de año y distancia
# presentan valores fuera de los límites del IQR.
print("Combinaciones de año y distancia atípicas:", len(resumen_atipicos))

# Mostramos las primeras 10 y las últimas 10
# para comparar distintas épocas históricas.
display(resumen_atipicos.head(10))
display(resumen_atipicos.tail(10))

Combinaciones de año y distancia atípicas: 11


,Year,Distance (km),Cantidad_ciclistas
0,1914,5380,53
1,1919,5560,10
2,1920,5503,22
3,1921,5485,37
4,1922,5375,38
5,1923,5386,29
6,1924,5425,60
7,1925,5440,49
8,1926,5745,40
9,1927,5398,39


,Year,Distance (km),Cantidad_ciclistas
1,1919,5560,10
2,1920,5503,22
3,1921,5485,37
4,1922,5375,38
5,1923,5386,29
6,1924,5425,60
7,1925,5440,49
8,1926,5745,40
9,1927,5398,39
10,1928,5476,40


### Tratamiento y justificación de los valores atípicos

Se aplicó el método del rango intercuartílico (IQR) a la columna `Distance (km)` para detectar valores extremos.

**Resultados:**
- Primer cuartil (Q1): 3,483 km.
- Tercer cuartil (Q3): 4,231 km.
- Rango intercuartílico: 748 km.
- Límite inferior: 2,361 km.
- Límite superior: 5,353 km.
- Registros atípicos detectados: 417 (4.03 % del total).

Los registros identificados corresponden a 11 ediciones históricas entre 1914 y 1928, con distancias comprendidas entre 5,375 y 5,745 km.

**Decisión de limpieza:** se conservaron los 417 registros, sin eliminarlos, sustituirlos ni modificar sus distancias.

**Justificación:** los valores extremos se concentran en determinadas ediciones históricas y pueden reflejar diferencias reales en los recorridos del Tour de Francia. El método IQR identifica valores estadísticamente inusuales, pero no demuestra que sean incorrectos. Por ello, se decidió conservarlos para evitar la pérdida de información histórica.

**Limitación:** las distancias se interpretaron según el dataset utilizado; no se verificaron individualmente contra registros oficiales externos.

## Crear una copia limpia de los ciclistas
## Limpieza de tiempos según el tipo de clasificación

In [13]:
# Creamos una copia para no modificar los datos originales.
ciclistas_limpios = ciclistas.copy()

# Identificamos los registros cuya clasificación
# no está expresada como tiempo.
sin_tiempo = ciclistas_limpios["ResultType"].isin(
    ["points", "no-results"]
)

# Convertimos la columna a un tipo entero que admite nulos.
ciclistas_limpios["TotalSeconds"] = (
    ciclistas_limpios["TotalSeconds"].astype("Int64")
)

# Sustituimos los ceros por valores faltantes únicamente
# en los registros sin clasificación basada en tiempo.
ciclistas_limpios.loc[
    sin_tiempo, "TotalSeconds"
] = pd.NA

# Comprobamos que se conservaron todos los ciclistas.
assert len(ciclistas_limpios) == len(ciclistas)

# Comprobamos que se marcaron exactamente 271 tiempos.
assert ciclistas_limpios["TotalSeconds"].isna().sum() == 271

# Verificamos que los tiempos válidos siguen siendo positivos.
assert (
    ciclistas_limpios.loc[
        ciclistas_limpios["ResultType"] == "time",
        "TotalSeconds"
    ] > 0
).all()

# Comparamos el resultado antes y después.
comparacion = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Tiempos iguales a cero",
        "Tiempos faltantes"
    ],
    "Antes": [
        len(ciclistas),
        int((ciclistas["TotalSeconds"] == 0).sum()),
        int(ciclistas["TotalSeconds"].isna().sum())
    ],
    "Después": [
        len(ciclistas_limpios),
        int((ciclistas_limpios["TotalSeconds"] == 0).sum()),
        int(ciclistas_limpios["TotalSeconds"].isna().sum())
    ]
})

display(comparacion)

,Indicador,Antes,Después
0,Total de registros,10337,10337
1,Tiempos iguales a cero,271,0
2,Tiempos faltantes,0,271


## Normalizar la columna Team

In [14]:
# Diccionario de variantes detectadas en la columna Team.
# La clave es el nombre original y el valor es el nombre unificado.
normalizacion_equipos = {
    "BELGIQUE": "Belgique",
    "ESPAGNE": "Espagne",
    "FDJEUX.COM": "FDJeux.com",
    "FESTINA WATCHES": "FESTINA Watches",
    "FRANCE": "France",
    "ILE DE FRANCE": "Ile de France",
    "ITALIE": "Italie",
    "OUEST": "Ouest",
    "SUD-EST": "Sud-Est",
    "SUD-OUEST": "Sud-Ouest",
    "SUISSE": "Suisse",
    "TEAM POLTI": "Team POLTI"
}

# Normalizamos únicamente los nombres que presentan
# las variantes de escritura identificadas.
ciclistas_limpios["Team"] = (
    ciclistas_limpios["Team"]
    .replace(normalizacion_equipos)
)

# Comprobamos la cantidad de nombres únicos
# después de aplicar la normalización.
print(
    "Equipos únicos después de normalizar:",
    ciclistas_limpios["Team"].nunique()
)

# Verificamos que no se hayan perdido registros.
assert len(ciclistas_limpios) == len(ciclistas)

print("Registros conservados:", len(ciclistas_limpios))

Equipos únicos después de normalizar: 690
Registros conservados: 10337


## Validación de diferencias de tiempo (GapSeconds)

In [15]:
# Agrupamos los registros por el tipo de clasificación.
# Revisamos cuántos tienen una diferencia de tiempo igual a cero
# y cuántos tienen una diferencia positiva.

resumen_gap = (
    ciclistas_limpios
    .groupby("ResultType")
    .agg(
        Total_registros=("GapSeconds", "size"),
        Gap_cero=("GapSeconds", lambda x: (x == 0).sum()),
        Gap_positivo=("GapSeconds", lambda x: (x > 0).sum()),
        Gap_nulo=("GapSeconds", lambda x: x.isna().sum())
    )
)

print("Distribución de GapSeconds por tipo de clasificación:")
display(resumen_gap)

# Revisamos específicamente los registros históricos
# que no tienen una clasificación basada en tiempo.
registros_historicos = ciclistas_limpios[
    ciclistas_limpios["ResultType"].isin(
        ["points", "no-results"]
    )
]

print("\nEjemplos de registros sin clasificación por tiempo:")
display(
    registros_historicos[
        ["Year", "Rank", "Rider", "ResultType",
         "Times", "Gap", "TotalSeconds", "GapSeconds"]
    ].head(15)
)

Distribución de GapSeconds por tipo de clasificación:


,Total_registros,Gap_cero,Gap_positivo,Gap_nulo
ResultType,,,,
no-results,74,74,0,0
points,197,197,0,0
time,10066,128,9938,0



Ejemplos de registros sin clasificación por tiempo:


,Year,Rank,Rider,ResultType,Times,Gap,TotalSeconds,GapSeconds
36,1905,1,LOUIS TROUSSELIER,no-results,00h 00' 00'',-,<NA>,0
37,1905,2,HIPPOLYTE AUCOUTURIER,no-results,00h 00' 00'',-,<NA>,0
38,1905,3,JEAN-BAPTISTE DORTIGNACQ,no-results,00h 00' 00'',-,<NA>,0
39,1905,4,EMILE GEORGET,no-results,00h 00' 00'',-,<NA>,0
40,1905,5,LUCIEN MAZAN DIT PETIT-BRETON,no-results,00h 00' 00'',-,<NA>,0
41,1905,6,AUGUSTINN RINGEVAL,no-results,00h 00' 00'',-,<NA>,0
42,1905,7,PAUL CHAUVET,no-results,00h 00' 00'',-,<NA>,0
43,1905,8,PHILIPPE PAUTRAT,no-results,00h 00' 00'',-,<NA>,0
44,1905,9,JULIEN GABORY,no-results,00h 00' 00'',-,<NA>,0
45,1905,10,JULIEN MAITRON,no-results,00h 00' 00'',-,<NA>,0


No encontramos valores nulos en GapSeconds antes de la limpieza.

• Los 271 ceros de points y no-results: no representan una diferencia de tiempo utilizable. Debemos convertirlos en valores faltantes (pd.NA).

• Los 128 ceros de time: pueden ser válidos, por ejemplo, cuando un ciclista tiene el mismo tiempo que el líder. Debemos conservarlos.

No sería correcto reemplazar todos los ceros por valores nulos.

## Limpiar GapSeconds
## Limpieza de diferencias de tiempo según ResultType

In [16]:
# Trabajamos sobre la copia limpia de los ciclistas.
# Los datos originales permanecen intactos.

# Convertimos GapSeconds a un entero que admite nulos.
ciclistas_limpios["GapSeconds"] = (
    ciclistas_limpios["GapSeconds"].astype("Int64")
)

# Identificamos las clasificaciones que no utilizan tiempos.
sin_diferencia_tiempo = ciclistas_limpios["ResultType"].isin(
    ["points", "no-results"]
)

# Marcamos como no disponibles las diferencias de tiempo
# que no son aplicables a esas clasificaciones.
ciclistas_limpios.loc[
    sin_diferencia_tiempo, "GapSeconds"
] = pd.NA

# Verificamos que no se eliminaron registros.
assert len(ciclistas_limpios) == len(ciclistas)

# Verificamos que exactamente 271 valores son nulos.
assert ciclistas_limpios["GapSeconds"].isna().sum() == 271

# Verificamos que conservamos los 128 ceros
# de las clasificaciones basadas en tiempo.
ceros_validos = (
    (ciclistas_limpios["ResultType"] == "time") &
    (ciclistas_limpios["GapSeconds"] == 0)
).sum()

assert ceros_validos == 128

# Comparamos los valores antes y después.
comparacion_gap = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Diferencias iguales a cero",
        "Diferencias faltantes"
    ],
    "Antes": [
        len(ciclistas),
        int((ciclistas["GapSeconds"] == 0).sum()),
        int(ciclistas["GapSeconds"].isna().sum())
    ],
    "Después": [
        len(ciclistas_limpios),
        int((ciclistas_limpios["GapSeconds"] == 0).sum()),
        int(ciclistas_limpios["GapSeconds"].isna().sum())
    ]
})

display(comparacion_gap)

,Indicador,Antes,Después
0,Total de registros,10337,10337
1,Diferencias iguales a cero,399,128
2,Diferencias faltantes,0,271


Hemos conseguido lo siguiente:
- Conservamos los 10,337 registros originales.
- Identificamos 271 registros históricos cuya clasificación no está basada en tiempos.
- Convertimos sus valores de tiempo no aplicables en pd.NA.
- Conservamos los 128 ceros válidos de GapSeconds.
- No modificamos los archivos CSV originales.

## Análisis histórico de valores faltantes en etapas

In [17]:
# Creamos una copia del dataset de etapas para trabajar
# sin modificar los datos originales.
etapas_limpias = etapas.copy()

# Seleccionamos las columnas relacionadas con
# ganadores y clasificaciones históricas.
columnas_clasificaciones = [
    "Winner of stage",
    "Yellow Jersey",
    "Green jersey",
    "Polka-dot jersey",
    "White jersey",
    "Leader"
]

# Calculamos el porcentaje de valores faltantes
# de cada columna en todo el dataset.
porcentaje_nulos = (
    etapas_limpias[columnas_clasificaciones]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

print("Porcentaje de valores faltantes por columna:")
display(porcentaje_nulos.to_frame("Porcentaje de nulos"))

# Para cada clasificación, identificamos el primer año
# y el último año con información disponible.
resumen_historico = []

for columna in columnas_clasificaciones:

    # Filtramos las filas donde existe un dato.
    registros_validos = etapas_limpias[
        etapas_limpias[columna].notna()
    ]

    # Guardamos la información histórica de la columna.
    resumen_historico.append({
        "Clasificación": columna,
        "Primer año con datos": registros_validos["Year"].min(),
        "Último año con datos": registros_validos["Year"].max(),
        "Registros disponibles": len(registros_validos),
        "Registros faltantes": etapas_limpias[columna].isna().sum()
    })

# Convertimos el resumen a un DataFrame.
resumen_historico = pd.DataFrame(resumen_historico)

print("\nDisponibilidad histórica de las clasificaciones:")
display(resumen_historico)

Porcentaje de valores faltantes por columna:


,Porcentaje de nulos
Winner of stage,1.40
Yellow Jersey,9.98
Green jersey,72.19
Polka-dot jersey,72.31
White jersey,73.25
Leader,93.63



Disponibilidad histórica de las clasificaciones:


,Clasificación,Primer año con datos,Último año con datos,Registros disponibles,Registros faltantes
0,Winner of stage,1903,2026,2400,34
1,Yellow Jersey,1919,2026,2191,243
2,Green jersey,1953,2026,677,1757
3,Polka-dot jersey,1933,2026,674,1760
4,White jersey,1977,2026,651,1783
5,Leader,1903,1914,155,2279


## Preparar la integración entre ciclistas y etapas
## Validación de la clave Year para integrar datasets

In [18]:
# Contamos los registros de ciclistas por año.
ciclistas_por_anio = (
    ciclistas_limpios.groupby("Year")
    .size()
    .rename("Cantidad_ciclistas")
)

# Contamos los registros de etapas por año.
etapas_por_anio = (
    etapas_limpias.groupby("Year")
    .size()
    .rename("Cantidad_etapas")
)

# Combinamos ambos conteos por año.
# No estamos uniendo todavía los registros individuales.
resumen_anual = pd.concat(
    [ciclistas_por_anio, etapas_por_anio],
    axis=1
)

# Identificamos años que solo aparecen en uno de los datasets.
anios_solo_ciclistas = (
    set(ciclistas_limpios["Year"])
    - set(etapas_limpias["Year"])
)

anios_solo_etapas = (
    set(etapas_limpias["Year"])
    - set(ciclistas_limpios["Year"])
)

# Mostramos las características de la clave Year.
print("Años distintos en ciclistas:",
      ciclistas_limpios["Year"].nunique())

print("Años distintos en etapas:",
      etapas_limpias["Year"].nunique())

print("Años solo en ciclistas:",
      sorted(anios_solo_ciclistas))

print("Años solo en etapas:",
      sorted(anios_solo_etapas))

# Revisamos las primeras y últimas ediciones disponibles.
print("\nPrimeros diez años:")
display(resumen_anual.head(10))

print("\nÚltimos diez años:")
display(resumen_anual.tail(10))

Años distintos en ciclistas: 113
Años distintos en etapas: 113
Años solo en ciclistas: []
Años solo en etapas: []

Primeros diez años:


,Cantidad_ciclistas,Cantidad_etapas
Year,,
1903,21,6
1904,15,6
1905,24,11
1906,14,14
1907,33,14
1908,36,15
1909,54,14
1910,41,15
1911,28,15



Últimos diez años:


,Cantidad_ciclistas,Cantidad_etapas
Year,,
2017,167,21
2018,145,21
2019,155,21
2020,146,21
2021,141,21
2022,135,24
2023,150,21
2024,141,21
2025,160,21


Conclusión: Year es una clave adecuada para relacionar las dos tablas, siempre que primero agrupemos las etapas por año. Así evitamos multiplicar artificialmente los registros de ciclistas.
## Integración de ciclistas y etapas por año
Integrar ciclistas y etapas con merge()

In [19]:
# Creamos un resumen de etapas por cada edición del Tour.
# Cada año tendrá una sola fila en esta nueva tabla.
resumen_etapas = (
    etapas_limpias
    .groupby("Year", as_index=False)
    .agg(
        Total_etapas=("Stages", "size"),
        Etapas_con_ganador=("Winner of stage", "count"),
        Prologos_registrados=("Stages", lambda x: (x == 0).sum())
    )
)

# Comprobamos que cada año aparece una sola vez.
assert resumen_etapas["Year"].is_unique

# Integramos el resumen anual con los ciclistas.
# many_to_one indica que muchos ciclistas pueden
# compartir un año, pero el resumen tiene una fila por año.
ciclistas_integrados = ciclistas_limpios.merge(
    resumen_etapas,
    on="Year",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Comprobamos que ningún ciclista se perdió o duplicó.
assert len(ciclistas_integrados) == len(ciclistas_limpios)

# Comprobamos que todos los ciclistas encontraron su año.
assert (ciclistas_integrados["_merge"] == "both").all()

# Eliminamos la columna auxiliar de verificación.
ciclistas_integrados = ciclistas_integrados.drop(
    columns="_merge"
)

# Mostramos las dimensiones antes y después.
print("Dimensiones originales:", ciclistas_limpios.shape)
print("Dimensiones integradas:", ciclistas_integrados.shape)

# Mostramos las nuevas columnas incorporadas.
display(
    ciclistas_integrados[
        [
            "Year",
            "Rider",
            "Rank",
            "TotalSeconds",
            "Total_etapas",
            "Etapas_con_ganador",
            "Prologos_registrados"
        ]
    ].head(10)
)

print("\nIntegración completada correctamente.")

Dimensiones originales: (10337, 14)
Dimensiones integradas: (10337, 17)


,Year,Rider,Rank,TotalSeconds,Total_etapas,Etapas_con_ganador,Prologos_registrados
0,1903,MAURICE GARIN,1,340394,6,6,0
1,1903,LUCIEN POTHIER,2,351155,6,6,0
2,1903,FERNAND AUGEREAU,3,356558,6,6,0
3,1903,RODOLPHE MULLER,4,357164,6,6,0
4,1903,JEAN-BAPTISTE FISCHER,5,358918,6,6,0
5,1903,MARCEL KERFF,6,365858,6,6,0
6,1903,DIT SAMSON JULIEN LOOTENS,7,374782,6,6,0
7,1903,GEORGES PASQUIER,8,381558,6,6,0
8,1903,FRANÇOIS BEAUGENDRE,9,383248,6,6,0
9,1903,ALOIS CATTEAU,10,390011,6,6,0



Integración completada correctamente.


Agregamos tres columnas, sin modificar la cantidad de ciclistas:
- Total_etapas: número de registros de etapas en esa edición.
- Etapas_con_ganador: cantidad de etapas con un ganador registrado.
- Prologos_registrados: cantidad de registros con Stages = 0.

En ciclismo, un prólogo es una prueba inicial que puede registrarse como etapa 0. Por eso, no debemos convertir esos ceros en valores faltantes ni eliminar esas filas. La interpretación es consistente con los datos, aunque todavía no hemos verificado la etiqueta oficial de cada etapa.

## Análisis de resultados mediante groupby()
Vamos a responder preguntas concretas utilizando los datos integrados:
- ¿Cuántos ciclistas registrados hay por edición?
- ¿Cuál es el tiempo promedio de los ciclistas por año?
- ¿Cuántas etapas están registradas en cada edición?

In [20]:
# Agrupamos los datos integrados por año.
# Cada fila del resultado representará una edición del Tour.

resumen_tour = (
    ciclistas_integrados
    .groupby("Year", as_index=False)
    .agg(
        # Cantidad de ciclistas registrados por edición.
        Total_ciclistas=("Rider", "size"),

        # Cantidad de ciclistas con un tiempo total disponible.
        Ciclistas_con_tiempo=("TotalSeconds", "count"),

        # Tiempo promedio de los ciclistas con datos válidos.
        Tiempo_promedio_segundos=("TotalSeconds", "mean"),

        # Menor tiempo registrado en la edición.
        Tiempo_minimo_segundos=("TotalSeconds", "min"),

        # Número de etapas registradas en el dataset.
        # Usamos first() porque todos los ciclistas
        # del mismo año tienen el mismo valor.
        Total_etapas=("Total_etapas", "first"),

        # Número de prólogos registrados en la edición.
        Prologos_registrados=("Prologos_registrados", "first")
    )
)

# Redondeamos el tiempo promedio a dos decimales.
resumen_tour["Tiempo_promedio_segundos"] = (
    resumen_tour["Tiempo_promedio_segundos"].round(2)
)

# Comprobamos que tenemos una fila por año.
assert resumen_tour["Year"].is_unique

# Comprobamos que los 10,337 ciclistas están contabilizados.
assert resumen_tour["Total_ciclistas"].sum() == 10337

# Comprobamos que solamente 10,066 tienen tiempo disponible.
assert resumen_tour["Ciclistas_con_tiempo"].sum() == 10066

# Mostramos los resultados de las primeras ediciones.
print("Primeras diez ediciones:")
display(resumen_tour.head(10))

# Mostramos los resultados de las últimas ediciones.
print("\nÚltimas diez ediciones:")
display(resumen_tour.tail(10))

# Mostramos las dimensiones del resumen.
print("\nDimensiones del resumen anual:", resumen_tour.shape)

Primeras diez ediciones:


,Year,Total_ciclistas,Ciclistas_con_tiempo,Tiempo_promedio_segundos,Tiempo_minimo_segundos,Total_etapas,Prologos_registrados
0,1903,21,21,414368.43,340394,6,0
1,1904,15,15,441867.73,345955,6,0
2,1905,24,0,<NA>,<NA>,11,0
3,1906,14,0,<NA>,<NA>,14,0
4,1907,33,0,<NA>,<NA>,14,0
5,1908,36,0,<NA>,<NA>,15,0
6,1909,54,0,<NA>,<NA>,14,0
7,1910,41,0,<NA>,<NA>,15,0
8,1911,28,0,<NA>,<NA>,15,0
9,1912,41,0,<NA>,<NA>,15,0



Últimas diez ediciones:


,Year,Total_ciclistas,Ciclistas_con_tiempo,Tiempo_promedio_segundos,Tiempo_minimo_segundos,Total_etapas,Prologos_registrados
103,2017,167,167,320410.92,310855,21,0
104,2018,145,145,308653.94,299833,21,0
105,2019,155,155,307546.35,298620,21,0
106,2020,146,146,326868.64,314405,21,0
107,2021,141,141,308184.45,298596,21,0
108,2022,135,135,297936.16,286400,24,0
109,2023,150,150,308525.48,295542,21,0
110,2024,141,141,314328.77,301136,21,0
111,2025,160,160,286831.95,273632,21,0
112,2026,158,158,281014.9,266186,21,0



Dimensiones del resumen anual: (113, 7)


## Tabla dinámica de ciclistas por año y tipo de clasificación
Crear una tabla dinámica con pivot_table()

In [21]:
# Construimos una tabla dinámica para contar ciclistas
# según el año y el tipo de clasificación.
tabla_dinamica = pd.pivot_table(
    ciclistas_integrados,

    # Cada fila representa una edición del Tour.
    index="Year",

    # Cada columna representa un tipo de clasificación.
    columns="ResultType",

    # Contamos los registros usando la columna Rider.
    values="Rider",

    # size cuenta la cantidad de registros por grupo.
    aggfunc="size",

    # Cuando no hay registros, mostramos cero.
    fill_value=0
)

# Añadimos una columna con el total de ciclistas por año.
tabla_dinamica["Total_ciclistas"] = (
    tabla_dinamica.sum(axis=1)
)

# Comprobamos que el total coincide con el dataset.
assert tabla_dinamica["Total_ciclistas"].sum() == 10337

# Comprobamos que tenemos una fila por edición.
assert len(tabla_dinamica) == 113

# Comparamos los totales con el resumen de groupby().
pd.testing.assert_series_equal(
    tabla_dinamica["Total_ciclistas"],
    resumen_tour.set_index("Year")["Total_ciclistas"],
    check_names=False
)

# Mostramos las primeras ediciones.
print("Primeras diez ediciones:")
display(tabla_dinamica.head(10))

# Mostramos las últimas ediciones.
print("\nÚltimas diez ediciones:")
display(tabla_dinamica.tail(10))

# Mostramos las dimensiones de la tabla dinámica.
print("\nDimensiones:", tabla_dinamica.shape)

Primeras diez ediciones:


ResultType,no-results,points,time,Total_ciclistas
Year,,,,
1903,0,0,21,21
1904,0,0,15,15
1905,24,0,0,24
1906,14,0,0,14
1907,0,33,0,33
1908,36,0,0,36
1909,0,54,0,54
1910,0,41,0,41
1911,0,28,0,28



Últimas diez ediciones:


ResultType,no-results,points,time,Total_ciclistas
Year,,,,
2017,0,0,167,167
2018,0,0,145,145
2019,0,0,155,155
2020,0,0,146,146
2021,0,0,141,141
2022,0,0,135,135
2023,0,0,150,150
2024,0,0,141,141
2025,0,0,160,160



Dimensiones: (113, 4)


## Validaciones finales de calidad de datos
Validaciones finales con assert

In [22]:
# 1. Comprobamos que conservamos todos los registros.
assert len(ciclistas_integrados) == 10337

# 2. Comprobamos que no existen duplicados exactos.
assert ciclistas_integrados.duplicated().sum() == 0

# 3. Comprobamos que todos los años tienen datos de etapas.
assert ciclistas_integrados["Total_etapas"].notna().all()

# 4. Comprobamos que las columnas incorporadas
# durante la integración no contienen valores negativos.
columnas_integradas = [
    "Total_etapas",
    "Etapas_con_ganador",
    "Prologos_registrados"
]

assert (
    ciclistas_integrados[columnas_integradas] >= 0
).all().all()

# 5. Comprobamos que los tiempos de clasificaciones
# por puntos o sin resultados están marcados como nulos.
sin_tiempo = ciclistas_integrados["ResultType"].isin(
    ["points", "no-results"]
)

assert ciclistas_integrados.loc[
    sin_tiempo, "TotalSeconds"
].isna().all()

assert ciclistas_integrados.loc[
    sin_tiempo, "GapSeconds"
].isna().all()

# 6. Comprobamos que los tiempos de las clasificaciones
# basadas en tiempo son positivos.
con_tiempo = (
    ciclistas_integrados["ResultType"] == "time"
)

assert (
    ciclistas_integrados.loc[
        con_tiempo, "TotalSeconds"
    ] > 0
).all()

# 7. Comprobamos que las diferencias de tiempo válidas
# son mayores o iguales a cero.
assert (
    ciclistas_integrados.loc[
        con_tiempo, "GapSeconds"
    ] >= 0
).all()

# 8. Comprobamos que todos los registros tienen año.
assert ciclistas_integrados["Year"].notna().all()

# 9. Comprobamos que los totales del resumen anual
# coinciden con el dataset integrado.
assert resumen_tour["Total_ciclistas"].sum() == len(
    ciclistas_integrados
)

# 10. Mostramos el resultado de las validaciones.
print("Todas las validaciones finales fueron exitosas.")
print("Registros:", len(ciclistas_integrados))
print("Columnas:", ciclistas_integrados.shape[1])
print(
    "Duplicados exactos:",
    ciclistas_integrados.duplicated().sum()
)
print(
    "Tiempos no disponibles:",
    ciclistas_integrados["TotalSeconds"].isna().sum()
)

Todas las validaciones finales fueron exitosas.
Registros: 10337
Columnas: 17
Duplicados exactos: 0
Tiempos no disponibles: 271


Hasta ahora hemos conseguido:
- Conservar los 10,337 registros de ciclistas.
- Integrar información de etapas, obteniendo 17 columnas.
- Corregir la interpretación de 271 tiempos no disponibles.
- Mantener los valores faltantes históricos que tienen una justificación.
- Verificar que no existen duplicados exactos y que la integración no perdió registros.
- Generar los análisis con groupby() y pivot_table().
## Exportación del dataset limpio e integrado a Parquet

In [23]:
# Creamos una carpeta para almacenar los resultados
# de la Tarea 6, sin modificar los CSV originales.
carpeta_resultados = Path("resultados_tarea6")
carpeta_resultados.mkdir(parents=True, exist_ok=True)

# Definimos la ruta del archivo Parquet final.
ruta_parquet = (
    carpeta_resultados / "tour_ciclistas_integrados.parquet"
)

# Exportamos los 10,337 registros y las 17 columnas.
# index=False evita guardar el índice como columna adicional.
ciclistas_integrados.to_parquet(
    ruta_parquet,
    index=False
)

# Verificamos que el archivo fue creado y no está vacío.
assert ruta_parquet.is_file()
assert ruta_parquet.stat().st_size > 0

# Leemos el Parquet para comprobar que se guardó correctamente.
datos_recuperados = pd.read_parquet(ruta_parquet)

# Comparamos el archivo recuperado con el DataFrame original.
# Esta prueba también comprueba valores, columnas y tipos de datos.
pd.testing.assert_frame_equal(
    datos_recuperados,
    ciclistas_integrados
)

# Mostramos los resultados.
print("Archivo Parquet exportado correctamente.")
print("Ruta:", ruta_parquet)
print("Filas:", len(datos_recuperados))
print("Columnas:", datos_recuperados.shape[1])
print(
    "Tamaño:",
    round(ruta_parquet.stat().st_size / 1024, 2),
    "KB"
)

Archivo Parquet exportado correctamente.
Ruta: resultados_tarea6\tour_ciclistas_integrados.parquet
Filas: 10337
Columnas: 17
Tamaño: 345.88 KB


## Tabla comparativa antes/después
## Comparación de calidad de datos antes y después de la limpieza
Documentamos los cambios realizados durante la limpieza y la integración.

In [24]:
# Construimos una tabla comparativa entre el dataset
# original de ciclistas y el dataset limpio e integrado.

comparacion_final = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Total de columnas",
        "Duplicados exactos",
        "TotalSeconds iguales a cero",
        "TotalSeconds faltantes",
        "GapSeconds iguales a cero",
        "GapSeconds faltantes",
        "Registros con información de etapas"
    ],

    # Valores del CSV original.
    "Antes": [
        len(ciclistas),
        ciclistas.shape[1],
        int(ciclistas.duplicated().sum()),
        int((ciclistas["TotalSeconds"] == 0).sum()),
        int(ciclistas["TotalSeconds"].isna().sum()),
        int((ciclistas["GapSeconds"] == 0).sum()),
        int(ciclistas["GapSeconds"].isna().sum()),
        0  # El CSV original no contiene las columnas integradas.
    ],

    # Valores del dataset limpio e integrado.
    "Después": [
        len(ciclistas_integrados),
        ciclistas_integrados.shape[1],
        int(ciclistas_integrados.duplicated().sum()),
        int((ciclistas_integrados["TotalSeconds"] == 0).sum()),
        int(ciclistas_integrados["TotalSeconds"].isna().sum()),
        int((ciclistas_integrados["GapSeconds"] == 0).sum()),
        int(ciclistas_integrados["GapSeconds"].isna().sum()),
        int(ciclistas_integrados["Total_etapas"].notna().sum())
    ]
})

# Mostramos la comparación.
display(comparacion_final)

# Comprobamos que no perdimos registros.
assert len(ciclistas_integrados) == len(ciclistas)

# Comprobamos que la integración añadió tres columnas.
assert ciclistas_integrados.shape[1] == ciclistas.shape[1] + 3

# Comprobamos que se identificaron 271 tiempos no disponibles.
assert ciclistas_integrados["TotalSeconds"].isna().sum() == 271

print("\nComparación generada correctamente.")

,Indicador,Antes,Después
0,Total de registros,10337,10337
1,Total de columnas,14,17
2,Duplicados exactos,0,0
3,TotalSeconds iguales a cero,271,0
4,TotalSeconds faltantes,0,271
5,GapSeconds iguales a cero,399,128
6,GapSeconds faltantes,0,271
7,Registros con información de etapas,0,10337



Comparación generada correctamente.


Aumentaron los valores faltantes, pero mejoró la interpretación de los datos.

Antes, los 271 registros históricos tenían ceros que podían confundirse con tiempos reales. Después, esos valores están correctamente identificados como no disponibles o no aplicables.

También aumentamos de 14 a 17 columnas gracias a la integración con las etapas, sin eliminar ni duplicar ciclistas.


### Comparación detallada de nulos y tipos de datos por columna

In [25]:
# Creamos una tabla con el diagnóstico de cada columna
# del dataset original de ciclistas masculinos.
diagnostico_antes = pd.DataFrame({
    "Columna": ciclistas.columns,
    "Tipo_antes": ciclistas.dtypes.astype(str).values,
    "Nulos_antes": ciclistas.isna().sum().values
})

# Creamos el mismo diagnóstico para el dataset
# después de la limpieza y la integración.
diagnostico_despues = pd.DataFrame({
    "Columna": ciclistas_integrados.columns,
    "Tipo_despues": ciclistas_integrados.dtypes.astype(str).values,
    "Nulos_despues": ciclistas_integrados.isna().sum().values
})

# Unimos los diagnósticos utilizando el nombre de la columna.
# outer permite incluir también las tres columnas nuevas
# que se incorporaron durante la integración.
comparacion_columnas = diagnostico_antes.merge(
    diagnostico_despues,
    on="Columna",
    how="outer",
    validate="one_to_one"
)

# Ordenamos las columnas para facilitar su interpretación.
comparacion_columnas = comparacion_columnas[
    [
        "Columna",
        "Tipo_antes",
        "Tipo_despues",
        "Nulos_antes",
        "Nulos_despues"
    ]
]

# Mostramos la tabla comparativa completa.
display(comparacion_columnas)

,Columna,Tipo_antes,Tipo_despues,Nulos_antes,Nulos_despues
0,B,str,str,10040.0,10040
1,Distance (km),int64,int64,0.0,0
2,Etapas_con_ganador,NaN,int64,NaN,0
3,Gap,str,str,0.0,0
4,GapSeconds,int64,Int64,0.0,271
5,Number of stages,int64,int64,0.0,0
6,P,str,str,10270.0,10270
7,Prologos_registrados,NaN,int64,NaN,0
8,Rank,int64,int64,0.0,0
9,ResultType,str,str,0.0,0


## Bitácora de transformaciones y decisiones de limpieza
### Bitácora de transformaciones — Tarea 6

**Proyecto:** Integración y limpieza de datos históricos del Tour de Francia.
**Fuente:** Repositorio LeTourDataSet, disponible en https://github.com/thomascamminady/LeTourDataSet
**Herramientas:** Python, Pandas, Jupyter Notebook y Parquet.

| Etapa | Operación realizada | Justificación y resultado |
|---|---|---|
| 1. Adquisición | Descarga de cuatro archivos CSV originales del repositorio. | Se conservaron los archivos fuente en `datos_tarea6`, permitiendo reproducir el análisis sin volver a descargarlos. |
| 2. Diagnóstico | Revisión de dimensiones, tipos de datos, valores faltantes, duplicados y estadísticas descriptivas. | Se identificaron 10,337 registros de ciclistas masculinos y 2,434 registros de etapas masculinas, además de dos datasets femeninos complementarios. |
| 3. Investigación de anomalías | Análisis de `TotalSeconds = 0`, `GapSeconds = 0` y `Stages = 0`. | Se distinguieron los valores históricos no aplicables de los ceros que pueden ser válidos. |
| 4. Valores atípicos | Aplicación del método IQR a `Distance (km)` e investigación por edición histórica. | Se detectaron 417 registros de 11 ediciones (1914–1928); se conservaron porque su carácter atípico no demuestra un error. |
| 5. Normalización de texto | Unificación de 12 variantes de escritura en la columna `Team`, conservando nombres legibles mediante un diccionario de reemplazos. | Los nombres únicos de equipos disminuyeron de 702 a 690. Se conservaron los 10,337 registros y los archivos originales permanecieron intactos. |
| 6. Limpieza de tiempos | Conversión de 271 valores de `TotalSeconds` a `pd.NA` cuando `ResultType` era `points` o `no-results`. | Se evitó interpretar como tiempos reales los ceros correspondientes a clasificaciones no basadas en tiempo. |
| 7. Limpieza de diferencias | Conversión de 271 valores de `GapSeconds` a `pd.NA`, conservando los 128 ceros de clasificaciones `time`. | Se preservaron las diferencias de tiempo válidas y se identificaron las no aplicables. |
| 8. Tratamiento de nulos históricos | Conservación de los valores faltantes en las columnas de ganadores y clasificaciones de etapas. | No se imputaron nombres ni resultados históricos sin evidencia suficiente. |
| 9. Integración | Agrupación de etapas por `Year` e integración mediante `merge(validate="many_to_one")`. | Se añadieron tres columnas a los ciclistas sin perder ni multiplicar registros. |
| 10. Análisis | Aplicación de `groupby()` y `pivot_table()`. | Se generaron resúmenes anuales y distribuciones de ciclistas por tipo de clasificación. |
| 11. Validación | Ejecución de comprobaciones con `assert`. | Se verificaron dimensiones, ausencia de duplicados exactos, coherencia de tiempos y conservación de registros. |
| 12. Exportación | Guardado del dataset integrado en formato Parquet y lectura posterior para verificarlo. | Se obtuvo `tour_ciclistas_integrados.parquet`, con 10,337 filas y 17 columnas. |

### Decisiones metodológicas

No se eliminaron registros únicamente por presentar valores faltantes o atípicos. Se conservaron las diferencias históricas entre ediciones y modalidades de clasificación. Los valores no aplicables se representaron mediante `pd.NA`, sin sustituirlos por promedios ni inventar información.

### Limitaciones

- Los datos cubren diferentes periodos históricos, por lo que las clasificaciones y distancias no son directamente comparables entre todas las ediciones.
- Algunas columnas de etapas tienen porcentajes elevados de valores faltantes.
- Los resúmenes reflejan los registros disponibles en la fuente, no necesariamente la totalidad de los participantes oficiales.
- Los tiempos promedio entre distintas ediciones pueden estar influidos por cambios en recorridos, distancias y condiciones de competición.
- Los cuatro archivos se descargaron y verificaron, pero el análisis principal se concentró en las dos tablas masculinas.